# LeetCode 922 — Sort Array By Parity II bằng Python

**Link đề:** https://leetcode.com/problems/sort-array-by-parity-ii/
**Độ khó:** Easy · **Chủ đề:** Array, Two Pointers, Sorting

```python
class Solution:
    def sortArrayByParityII(self, nums: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **tính chẵn lẻ** (parity) của số và của **chỉ số** (index).
2. Biết dùng `% 2`, `range(start, stop, step)`, slicing có bước `nums[::2]`, gán slice.
3. Đi từ cách tách hai danh sách → mảng kết quả với hai con trỏ ghi → **hai con trỏ hoán đổi tại chỗ** (follow-up in-place).
4. Viết **hàm kiểm tra đáp án** (validator) cho bài có **nhiều đáp án đúng**.
5. Trace, chứng minh tính đúng, phân tích Time/Space, luyện các biến thể (LeetCode 905...).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — số chẵn ở vị trí chẵn, số lẻ ở vị trí lẻ

Cho mảng `nums` có **đúng một nửa là số chẵn, một nửa là số lẻ**. Sắp xếp lại sao cho:
- Khi `nums[i]` lẻ thì `i` lẻ.
- Khi `nums[i]` chẵn thì `i` chẵn.

Trả về **bất kỳ** mảng nào thỏa điều kiện.

| `nums` | Một output hợp lệ | Các output khác cũng đúng |
|---|---|---|
| `[4,2,5,7]` | `[4,5,2,7]` | `[4,7,2,5]`, `[2,5,4,7]`, `[2,7,4,5]` |
| `[2,3]` | `[2,3]` | — |

**Constraints:** `2 <= nums.length <= 2·10^4`, độ dài chẵn, đúng nửa số chẵn, `0 <= nums[i] <= 1000`.

**Follow up:** Làm **tại chỗ** (in-place) được không?

Vì có nhiều đáp án đúng, khi test ta **không so sánh với một mảng cố định** mà phải kiểm tra **tính chất**.

In [1]:
nums = [4, 2, 5, 7]
answer = [4, 5, 2, 7]
for i, x in enumerate(answer):
    print(f"index {i} ({'chẵn' if i % 2 == 0 else 'lẻ'}) chứa {x} ({'chẵn' if x % 2 == 0 else 'lẻ'})")

index 0 (chẵn) chứa 4 (chẵn)
index 1 (lẻ) chứa 5 (lẻ)
index 2 (chẵn) chứa 2 (chẵn)
index 3 (lẻ) chứa 7 (lẻ)


## 2. Nền tảng Python cần dùng

### 2.1. Chẵn lẻ với `% 2`

- `x % 2 == 0` → chẵn, `x % 2 == 1` → lẻ (với `x >= 0`).
- Mẹo: "`x` và `i` cùng tính chẵn lẻ" ⇔ `x % 2 == i % 2`.

Đây chính là điều kiện của bài: **mọi `i`, `nums[i] % 2 == i % 2`**.

In [2]:
for x in [0, 1, 2, 7, 1000]:
    print(x, "% 2 =", x % 2)
print("Cùng tính chẵn lẻ (5, 3)?", 5 % 2 == 3 % 2)
print("Cùng tính chẵn lẻ (4, 3)?", 4 % 2 == 3 % 2)

0 % 2 = 0
1 % 2 = 1
2 % 2 = 0
7 % 2 = 1
1000 % 2 = 0
Cùng tính chẵn lẻ (5, 3)? True
Cùng tính chẵn lẻ (4, 3)? False


### 2.2. Hàm kiểm tra đáp án (validator)

Ta viết hàm `is_valid(original, answer)` kiểm tra:
1. `answer` là **hoán vị** của `original` (cùng các phần tử, cùng số lần) → so `sorted(...)`.
2. Mọi vị trí thỏa `answer[i] % 2 == i % 2`.

`all(...)` trả `True` khi mọi điều kiện con đúng.

In [3]:
def is_valid(original: list[int], answer: list[int]) -> bool:
    if sorted(original) != sorted(answer):
        return False
    return all(x % 2 == i % 2 for i, x in enumerate(answer))

print(is_valid([4,2,5,7], [4,5,2,7]))   # True
print(is_valid([4,2,5,7], [2,7,4,5]))   # True
print(is_valid([4,2,5,7], [4,2,5,7]))   # False: 2 ở index 1 (lẻ)
print(is_valid([4,2,5,7], [4,5,4,7]))   # False: không phải hoán vị

True
True
False
False


### 2.3. `range(start, stop, step)` — duyệt các vị trí chẵn / lẻ

- `range(0, n, 2)` → `0, 2, 4, ...` (vị trí chẵn).
- `range(1, n, 2)` → `1, 3, 5, ...` (vị trí lẻ).

In [4]:
n = 8
print("chẵn:", list(range(0, n, 2)))
print("lẻ  :", list(range(1, n, 2)))

chẵn: [0, 2, 4, 6]
lẻ  : [1, 3, 5, 7]


### 2.4. Slicing có bước và **gán slice**

- `a[::2]` lấy các phần tử ở vị trí chẵn; `a[1::2]` lấy vị trí lẻ.
- Có thể **gán** vào slice: `a[::2] = [...]` thay toàn bộ các vị trí chẵn (độ dài phải khớp).

In [5]:
a = [10, 11, 12, 13, 14, 15]
print("a[::2]  =", a[::2])
print("a[1::2] =", a[1::2])
res = [None] * 6
res[::2] = ["E0", "E1", "E2"]
res[1::2] = ["O0", "O1", "O2"]
print(res)

a[::2]  = [10, 12, 14]
a[1::2] = [11, 13, 15]
['E0', 'O0', 'E1', 'O1', 'E2', 'O2']


`[None] * 6` tạo list 6 phần tử `None` — một "khung" để điền dần.

### 2.5. List comprehension có điều kiện

`[x for x in nums if x % 2 == 0]` → lấy các số chẵn theo thứ tự xuất hiện.

In [6]:
nums = [4, 2, 5, 7]
evens = [x for x in nums if x % 2 == 0]
odds = [x for x in nums if x % 2 == 1]
print("evens =", evens, "| odds =", odds)

evens = [4, 2] | odds = [5, 7]


## 3. Cách trực giác — tách hai danh sách rồi xen kẽ

1. Tách `evens` và `odds`.
2. Tạo kết quả: vị trí `0` lấy `evens[0]`, vị trí `1` lấy `odds[0]`, vị trí `2` lấy `evens[1]`...

Vì đề đảm bảo đúng nửa chẵn nửa lẻ, hai danh sách dài bằng nhau = `n/2`.

In [7]:
def parity_ii_split(nums: list[int]) -> list[int]:
    evens = [x for x in nums if x % 2 == 0]
    odds = [x for x in nums if x % 2 == 1]
    result = []
    for e, o in zip(evens, odds):
        result.append(e)
        result.append(o)
    return result

for nums in [[4, 2, 5, 7], [2, 3], [3, 1, 4, 2, 6, 5]]:
    out = parity_ii_split(nums)
    print(nums, "->", out, "| hợp lệ:", is_valid(nums, out))

[4, 2, 5, 7] -> [4, 5, 2, 7] | hợp lệ: True
[2, 3] -> [2, 3] | hợp lệ: True
[3, 1, 4, 2, 6, 5] -> [4, 3, 2, 1, 6, 5] | hợp lệ: True


### 3.1. `zip` — đi song song hai danh sách

`zip(a, b)` ghép từng cặp `(a[0], b[0]), (a[1], b[1]), ...`, dừng ở danh sách ngắn hơn.

In [8]:
print(list(zip([4, 2], [5, 7])))
print(list(zip([1, 2, 3], ["x", "y"])))   # dừng ở độ dài ngắn hơn

[(4, 5), (2, 7)]
[(1, 'x'), (2, 'y')]


### 3.2. Phiên bản slice gọn

Dùng gán slice từ mục 2.4: `res[::2] = evens; res[1::2] = odds`.

In [9]:
def parity_ii_slice(nums: list[int]) -> list[int]:
    res = [0] * len(nums)
    res[::2] = [x for x in nums if x % 2 == 0]
    res[1::2] = [x for x in nums if x % 2 == 1]
    return res

print(parity_ii_slice([4, 2, 5, 7]), parity_ii_slice([3, 1, 4, 2, 6, 5]))

[4, 5, 2, 7] [4, 3, 2, 1, 6, 5]


## 4. Nút thắt và insight

Cách tách dùng thêm `O(n)` bộ nhớ (hai list + kết quả) và duyệt mảng 2–3 lần. Follow-up hỏi: **làm tại chỗ**?

**Quan sát then chốt:**
- Số vị trí chẵn = số số chẵn = `n/2`. Nên nếu ở một vị trí chẵn có một **số lẻ "lạc chỗ"**, thì chắc chắn ở đâu đó có một vị trí lẻ chứa **số chẵn "lạc chỗ"** (đếm số lượng: các vị trí chẵn thiếu bao nhiêu số chẵn thì các vị trí lẻ thừa bấy nhiêu số chẵn).
- **Đổi chỗ hai phần tử lạc chỗ đó** sửa đúng cả hai vị trí trong một lần.

**Thuật toán hai con trỏ:**
- `i` chỉ duyệt vị trí chẵn `0, 2, 4, ...`
- `j` chỉ duyệt vị trí lẻ `1, 3, 5, ...`
- Với mỗi `i` mà `nums[i]` lẻ: tiến `j` cho đến khi `nums[j]` chẵn, rồi đổi chỗ.
- `j` **không bao giờ lùi** → tổng công việc `O(n)`.

In [10]:
# Minh họa quan sát "đếm lạc chỗ" trên một mảng
nums = [3, 1, 4, 2, 6, 5]
bad_even_pos = [i for i in range(0, len(nums), 2) if nums[i] % 2 == 1]
bad_odd_pos = [j for j in range(1, len(nums), 2) if nums[j] % 2 == 0]
print("vị trí chẵn chứa số lẻ:", bad_even_pos)
print("vị trí lẻ chứa số chẵn:", bad_odd_pos)
print("Số lượng bằng nhau?", len(bad_even_pos) == len(bad_odd_pos))

vị trí chẵn chứa số lẻ: [0]
vị trí lẻ chứa số chẵn: [3]
Số lượng bằng nhau? True


## 5. Lời giải chính — hai con trỏ hoán đổi tại chỗ

**Mảnh 1:** `j = 1` (con trỏ vị trí lẻ).

**Mảnh 2:** `for i in range(0, n, 2):` — duyệt vị trí chẵn.

**Mảnh 3:** nếu `nums[i] % 2 == 1` (số lẻ lạc chỗ):
- `while nums[j] % 2 == 1: j += 2` — tìm số chẵn lạc chỗ ở vị trí lẻ.
- Đổi chỗ `nums[i], nums[j]`.

**Mảnh 4:** trả `nums`.

Vì sao `while` không chạy quá mảng? Nhờ quan sát ở mục 4: mỗi khi có số lẻ lạc ở vị trí chẵn, **chắc chắn còn** một số chẵn lạc ở vị trí lẻ phía sau `j`.

In [11]:
def parity_ii_inplace(nums: list[int]) -> list[int]:
    n = len(nums)
    j = 1
    for i in range(0, n, 2):
        if nums[i] % 2 == 1:
            while nums[j] % 2 == 1:
                j += 2
            nums[i], nums[j] = nums[j], nums[i]
    return nums

for nums in [[4, 2, 5, 7], [2, 3], [3, 1, 4, 2, 6, 5]]:
    original = nums[:]                 # sao chép vì hàm sửa tại chỗ
    out = parity_ii_inplace(nums)
    print(original, "->", out, "| hợp lệ:", is_valid(original, out))

[4, 2, 5, 7] -> [4, 5, 2, 7] | hợp lệ: True
[2, 3] -> [2, 3] | hợp lệ: True
[3, 1, 4, 2, 6, 5] -> [2, 1, 4, 3, 6, 5] | hợp lệ: True


`nums[:]` tạo **bản sao nông** (shallow copy) của list. Cần thiết vì hàm sửa trực tiếp `nums`; nếu không sao chép, ta mất mảng gốc để kiểm tra.

In [12]:
a = [1, 2]
alias = a          # cùng một list
copy = a[:]        # list mới
a[0] = 99
print("alias:", alias, "| copy:", copy)

alias: [99, 2] | copy: [1, 2]


## 6. TRACE từng bước

Ví dụ `nums = [3, 1, 4, 2, 6, 5]` (n = 6).

| i | nums[i] | Lạc chỗ? | j trước | j sau khi tìm | Đổi | nums sau |
|:---:|:---:|:---:|:---:|:---:|---|---|
| 0 | 3 | lẻ ở chẵn → có | 1 | 3 (nums[1]=1 lẻ → j=3, nums[3]=2 chẵn) | 3↔2 | `[2,1,4,3,6,5]` |
| 2 | 4 | không | 3 | 3 | — | `[2,1,4,3,6,5]` |
| 4 | 6 | không | 3 | 3 | — | `[2,1,4,3,6,5]` |

Kết quả `[2,1,4,3,6,5]`: chẵn ở 0,2,4; lẻ ở 1,3,5. ✓

In [13]:
def trace_parity_ii(nums):
    nums = nums[:]
    n = len(nums)
    j = 1
    print("Bắt đầu:", nums)
    for i in range(0, n, 2):
        if nums[i] % 2 == 0:
            print(f"i={i}: nums[i]={nums[i]} chẵn -> đúng chỗ")
            continue
        print(f"i={i}: nums[i]={nums[i]} lẻ -> lạc chỗ, tìm j", end="")
        while nums[j] % 2 == 1:
            print(f" [j={j}: {nums[j]} lẻ, bỏ qua]", end="")
            j += 2
        print(f" -> j={j}: {nums[j]} chẵn")
        nums[i], nums[j] = nums[j], nums[i]
        print(f"     đổi chỗ -> {nums}")
    print("Kết quả:", nums)
    return nums

trace_parity_ii([3, 1, 4, 2, 6, 5])

Bắt đầu: [3, 1, 4, 2, 6, 5]
i=0: nums[i]=3 lẻ -> lạc chỗ, tìm j [j=1: 1 lẻ, bỏ qua] -> j=3: 2 chẵn
     đổi chỗ -> [2, 1, 4, 3, 6, 5]
i=2: nums[i]=4 chẵn -> đúng chỗ
i=4: nums[i]=6 chẵn -> đúng chỗ
Kết quả: [2, 1, 4, 3, 6, 5]


[2, 1, 4, 3, 6, 5]

In [14]:
# Trường hợp "xấu": mọi vị trí đều lạc chỗ
trace_parity_ii([1, 2, 3, 4, 5, 6, 7, 8])

Bắt đầu: [1, 2, 3, 4, 5, 6, 7, 8]
i=0: nums[i]=1 lẻ -> lạc chỗ, tìm j -> j=1: 2 chẵn
     đổi chỗ -> [2, 1, 3, 4, 5, 6, 7, 8]
i=2: nums[i]=3 lẻ -> lạc chỗ, tìm j [j=1: 1 lẻ, bỏ qua] -> j=3: 4 chẵn
     đổi chỗ -> [2, 1, 4, 3, 5, 6, 7, 8]
i=4: nums[i]=5 lẻ -> lạc chỗ, tìm j [j=3: 3 lẻ, bỏ qua] -> j=5: 6 chẵn
     đổi chỗ -> [2, 1, 4, 3, 6, 5, 7, 8]
i=6: nums[i]=7 lẻ -> lạc chỗ, tìm j [j=5: 5 lẻ, bỏ qua] -> j=7: 8 chẵn
     đổi chỗ -> [2, 1, 4, 3, 6, 5, 8, 7]
Kết quả: [2, 1, 4, 3, 6, 5, 8, 7]


[2, 1, 4, 3, 6, 5, 8, 7]

`continue` bỏ qua phần còn lại của vòng hiện tại, sang `i` tiếp theo. `print(..., end="")` in không xuống dòng để gom thông tin trên một dòng.

## 7. Các lời giải khác và bảng so sánh

### 7.1. Hai con trỏ **ghi** vào mảng kết quả

Duyệt `nums` một lần; số chẵn ghi vào `res[even_idx]` rồi `even_idx += 2`; số lẻ ghi vào `res[odd_idx]` rồi `odd_idx += 2`.

In [15]:
def parity_ii_write_pointers(nums: list[int]) -> list[int]:
    res = [0] * len(nums)
    even_idx, odd_idx = 0, 1
    for x in nums:
        if x % 2 == 0:
            res[even_idx] = x
            even_idx += 2
        else:
            res[odd_idx] = x
            odd_idx += 2
    return res

print(parity_ii_write_pointers([4, 2, 5, 7]), parity_ii_write_pointers([3, 1, 4, 2, 6, 5]))

[4, 5, 2, 7] [4, 3, 2, 1, 6, 5]


### 7.2. Sắp xếp theo khóa chẵn lẻ rồi xen kẽ

`sorted(nums, key=lambda x: x % 2)` đưa số chẵn (khóa 0) lên trước, số lẻ (khóa 1) ra sau, **giữ thứ tự tương đối** (sắp xếp của Python là **ổn định** — stable). Sau đó nửa đầu là chẵn, nửa sau là lẻ → xen kẽ. Tốn `O(n log n)` — không cần thiết, nhưng là cách nhìn hay.

In [16]:
def parity_ii_sort_key(nums: list[int]) -> list[int]:
    arr = sorted(nums, key=lambda x: x % 2)
    half = len(arr) // 2
    res = [0] * len(arr)
    res[::2] = arr[:half]
    res[1::2] = arr[half:]
    return res

print(sorted([3, 1, 4, 2, 6, 5], key=lambda x: x % 2))
print(parity_ii_sort_key([3, 1, 4, 2, 6, 5]))

[4, 2, 6, 3, 1, 5]
[4, 3, 2, 1, 6, 5]


### 7.3. Hai con trỏ cùng tiến (biến thể in-place đối xứng)

Thay vì `for i`, dùng `while i < n and j < n`: tiến `i` qua các vị trí chẵn đúng, tiến `j` qua các vị trí lẻ đúng; khi cả hai dừng ở chỗ sai → đổi chỗ.

In [17]:
def parity_ii_two_while(nums: list[int]) -> list[int]:
    n = len(nums)
    i, j = 0, 1
    while i < n and j < n:
        if nums[i] % 2 == 0:
            i += 2
        elif nums[j] % 2 == 1:
            j += 2
        else:
            nums[i], nums[j] = nums[j], nums[i]
            i += 2
            j += 2
    return nums

print(parity_ii_two_while([1, 2, 3, 4, 5, 6, 7, 8]))

[2, 1, 4, 3, 6, 5, 8, 7]


### 7.4. Bảng so sánh

`n = len(nums)`.

| Cách | Time | Space phụ | Tại chỗ? | Ghi chú |
|---|---|---|:---:|---|
| Tách + zip | `O(n)` | `O(n)` | ✗ | dễ nghĩ nhất |
| Tách + gán slice | `O(n)` | `O(n)` | ✗ | gọn |
| Hai con trỏ ghi | `O(n)` | `O(n)` (mảng kết quả) | ✗ | duyệt 1 lần |
| Sort theo khóa | `O(n log n)` | `O(n)` | ✗ | minh họa stable sort |
| **Hai con trỏ hoán đổi (for + while)** | `O(n)` | `O(1)` | ✓ | trả lời follow-up |
| Hai while đối xứng | `O(n)` | `O(1)` | ✓ | cùng ý, viết khác |

## 8. Code nộp LeetCode

In [18]:
class Solution:
    def sortArrayByParityII(self, nums: list[int]) -> list[int]:
        j = 1
        for i in range(0, len(nums), 2):
            if nums[i] % 2 == 1:
                while nums[j] % 2 == 1:
                    j += 2
                nums[i], nums[j] = nums[j], nums[i]
        return nums

sol = Solution()
for nums in [[4, 2, 5, 7], [2, 3]]:
    original = nums[:]
    out = sol.sortArrayByParityII(nums)
    assert is_valid(original, out), (original, out)
    print(original, "->", out)
print("Solution PASS ví dụ đề bài (kiểm tra bằng validator)")

[4, 2, 5, 7] -> [4, 5, 2, 7]
[2, 3] -> [2, 3]
Solution PASS ví dụ đề bài (kiểm tra bằng validator)


## 9. Vì sao thuật toán đúng?

**Invariant (đầu mỗi vòng `for` với chỉ số `i`):**
1. Mọi vị trí chẵn `< i` chứa số chẵn.
2. Mọi vị trí lẻ `< j` chứa số lẻ.

**Giữ invariant:**
- Nếu `nums[i]` chẵn: không làm gì, invariant 1 mở rộng thêm `i`.
- Nếu `nums[i]` lẻ: vòng `while` bỏ qua các vị trí lẻ đã đúng (chứa số lẻ), dừng ở `j` chứa số chẵn. Đổi chỗ → `nums[i]` chẵn, `nums[j]` lẻ. Cả hai vị trí đúng.

**`while` không vượt mảng:** Khi gặp `nums[i]` lẻ, xét đoạn chưa xử lý. Các vị trí chẵn `>= i` có `n/2 - i/2` ô; số số chẵn chưa đặt cũng là `n/2 - i/2` (vì đã có đúng `i/2` số chẵn ở vị trí chẵn `< i`, và vị trí lẻ `< j` chỉ chứa số lẻ). Ô `i` đang chứa số lẻ nên ít nhất một số chẵn chưa đặt nằm ở **vị trí lẻ `>= j`**. Vậy `while` dừng trước khi hết mảng.

**Kết thúc:** sau `for`, mọi vị trí chẵn chứa số chẵn → còn lại `n/2` số lẻ nằm ở `n/2` vị trí lẻ. ∎

**Mảng vẫn là hoán vị** của ban đầu vì ta chỉ đổi chỗ.

In [19]:
# Kiểm chứng invariant trong lúc chạy
def checked_inplace(nums):
    n = len(nums); j = 1
    for i in range(0, n, 2):
        assert all(nums[p] % 2 == 0 for p in range(0, i, 2))
        assert all(nums[q] % 2 == 1 for q in range(1, j, 2))
        if nums[i] % 2 == 1:
            while nums[j] % 2 == 1:
                j += 2
            nums[i], nums[j] = nums[j], nums[i]
    return nums

print(checked_inplace([1, 3, 5, 7, 2, 4, 6, 8]), "- invariant luôn đúng")

[4, 3, 8, 7, 2, 1, 6, 5] - invariant luôn đúng


## 10. Phân tích Time / Space

`n = len(nums)`.

- **Time `O(n)`:** `i` đi qua `n/2` vị trí chẵn; `j` chỉ **tăng**, đi qua tối đa `n/2` vị trí lẻ **trong toàn bộ thuật toán** (không reset). Tổng số bước `≤ n`. Đây là phân tích **khấu hao** (amortized): vòng `while` lồng trong `for` nhưng không làm thành `O(n²)`.
- **Space `O(1)`:** chỉ hai biến chỉ số, sửa trực tiếp `nums` (không tính mảng trả về chính là input).

In [20]:
# Đếm số bước của j để thấy tổng tuyến tính
def count_steps(nums):
    n = len(nums); j = 1; steps = 0
    for i in range(0, n, 2):
        if nums[i] % 2 == 1:
            while nums[j] % 2 == 1:
                j += 2; steps += 1
            nums[i], nums[j] = nums[j], nums[i]
    return steps

for n in [10, 100, 1000, 10000]:
    worst = list(range(1, n, 2)) + list(range(0, n, 2))   # toàn số lẻ trước, chẵn sau
    print(f"n={n:>5}: số lần j tiến = {count_steps(worst)} (≤ n/2 = {n // 2})")

n=   10: số lần j tiến = 4 (≤ n/2 = 5)
n=  100: số lần j tiến = 49 (≤ n/2 = 50)
n= 1000: số lần j tiến = 499 (≤ n/2 = 500)
n=10000: số lần j tiến = 4999 (≤ n/2 = 5000)


## 11. Bộ test — mẫu đề, biên, đối chiếu

Vì nhiều đáp án đúng, mọi test dùng `is_valid`. Nhóm test:
1. Ví dụ đề.
2. Biên: `n = 2` cả hai thứ tự `[2,3]`, `[3,2]`.
3. Đã đúng sẵn; toàn bộ lạc chỗ; số 0; số trùng.
4. Ngẫu nhiên có seed.

In [21]:
tests = [
    [4, 2, 5, 7],
    [2, 3],
    [3, 2],
    [0, 1],
    [1, 0, 3, 2],
    [2, 1, 4, 3, 6, 5],
    [1, 3, 5, 7, 0, 2, 4, 6],
    [7, 7, 8, 8],
    [1000, 999, 1, 0],
]
solvers = [parity_ii_split, parity_ii_slice, parity_ii_inplace, parity_ii_write_pointers,
           parity_ii_sort_key, parity_ii_two_while, Solution().sortArrayByParityII]
for nums in tests:
    for f in solvers:
        out = f(nums[:])
        assert is_valid(nums, out), (f.__name__, nums, out)
print(f"PASS {len(tests)} test x {len(solvers)} cách")

PASS 9 test x 7 cách


In [22]:
import random
rng = random.Random(922)
for _ in range(500):
    half = rng.randint(1, 15)
    nums = [rng.randrange(0, 1001, 2) for _ in range(half)] + [rng.randrange(1, 1001, 2) for _ in range(half)]
    rng.shuffle(nums)
    for f in solvers:
        assert is_valid(nums, f(nums[:])), (f.__name__, nums)
print("PASS 500 test ngẫu nhiên")

PASS 500 test ngẫu nhiên


`rng.randrange(0, 1001, 2)` chọn ngẫu nhiên một số **chẵn** trong `[0, 1000]`; `randrange(1, 1001, 2)` chọn số **lẻ**.

## 12. Lỗi người mới hay gặp

### Lỗi 1: So sánh output với **một** đáp án cố định

In [23]:
out = parity_ii_inplace([4, 2, 5, 7])
print("out =", out, "| == [2,7,4,5] (một đáp án đúng khác)?", out == [2, 7, 4, 5], "| hợp lệ?", is_valid([4, 2, 5, 7], out))

out = [4, 5, 2, 7] | == [2,7,4,5] (một đáp án đúng khác)? False | hợp lệ? True


**Vì sao sai:** đề chấp nhận **bất kỳ** đáp án hợp lệ; so sánh cứng sẽ báo sai nhầm. **Sửa:** dùng validator.

### Lỗi 2: Reset `j = 1` bên trong vòng `for`

In [24]:
def slow_reset_j(nums):
    n = len(nums); steps = 0
    for i in range(0, n, 2):
        if nums[i] % 2 == 1:
            j = 1                       # reset mỗi lần -> O(n^2)
            while nums[j] % 2 == 1:
                j += 2; steps += 1
            nums[i], nums[j] = nums[j], nums[i]
    return steps

n = 2000
worst = list(range(1, n, 2)) + list(range(0, n, 2))
print("reset j: số bước =", slow_reset_j(worst[:]), "| giữ j:", count_steps(worst[:]))

reset j: số bước = 374750 | giữ j: 999


**Vì sao sai:** vẫn cho đáp án đúng nhưng `O(n²)`; với `n = 2·10^4` chậm rõ. **Sửa:** khai báo `j` **ngoài** vòng `for`.

### Lỗi 3: Dùng `j += 1` thay `j += 2`

`j` phải luôn là vị trí **lẻ**. `j += 1` khiến `j` nhảy vào vị trí chẵn và đổi chỗ phá hỏng vị trí chẵn đã đúng.

In [25]:
def wrong_step_one(nums):
    j = 1
    for i in range(0, len(nums), 2):
        if nums[i] % 2 == 1:
            while nums[j] % 2 == 1:
                j += 1                  # SAI: có thể rơi vào vị trí chẵn
            nums[i], nums[j] = nums[j], nums[i]
    return nums

src = [0, 1, 2, 3, 5, 4]
out = wrong_step_one(src[:])
print(out, "| hợp lệ?", is_valid(src, out))

[0, 1, 5, 3, 2, 4] | hợp lệ? False


### Lỗi 4: Quên sao chép khi test hàm sửa tại chỗ

`out = f(nums)` rồi `is_valid(nums, out)` — `nums` đã bị sửa, ta so một mảng với chính nó, test luôn "qua" phần hoán vị. **Sửa:** `original = nums[:]` trước khi gọi.

### Lỗi 5: Kiểm tra lẻ bằng `x % 2 == -1` hoặc `x & 1` với số âm

Với bài này `nums[i] >= 0` nên không sao. Nhưng nhớ: trong Python `-3 % 2 == 1` (luôn không âm), khác C/Java (`-1`).

In [26]:
print(-3 % 2, -4 % 2, (-3) & 1)

1 0 1


## 13. Mẹo nhận dạng pattern

- "Đặt loại A vào nhóm vị trí P, loại B vào nhóm vị trí Q", số lượng khớp nhau → **hai con trỏ, mỗi con trỏ đi trên một nhóm vị trí**, đổi chỗ khi cả hai cùng lạc chỗ.
- Con trỏ "tìm kiếm" **không bao giờ lùi** → `while` lồng `for` vẫn `O(n)` (khấu hao).
- Bài có nhiều đáp án → viết **validator** kiểm tra tính chất thay vì so sánh cố định.

Bài cùng họ: 905 Sort Array By Parity, 75 Sort Colors (Dutch National Flag), 283 Move Zeroes, 2149 Rearrange Array Elements by Sign.

## 14. Biến thể

### 14.1. LeetCode 905 — chẵn lên trước, lẻ ra sau

Hai con trỏ từ hai đầu: `l` tìm số lẻ từ trái, `r` tìm số chẵn từ phải, đổi chỗ.

In [27]:
def sort_by_parity_905(nums):
    l, r = 0, len(nums) - 1
    while l < r:
        if nums[l] % 2 == 0:
            l += 1
        elif nums[r] % 2 == 1:
            r -= 1
        else:
            nums[l], nums[r] = nums[r], nums[l]
            l += 1; r -= 1
    return nums

print(sort_by_parity_905([3, 1, 2, 4]))

[4, 2, 1, 3]


### 14.2. LeetCode 2149 — xen kẽ dương/âm, giữ thứ tự

Chính là cách "hai con trỏ ghi" (mục 7.1) với điều kiện dương/âm thay cho chẵn/lẻ.

In [28]:
def rearrange_by_sign(nums):
    res = [0] * len(nums)
    pos, neg = 0, 1
    for x in nums:
        if x > 0:
            res[pos] = x; pos += 2
        else:
            res[neg] = x; neg += 2
    return res

print(rearrange_by_sign([3, 1, -2, -5, 2, -4]))   # [3,-2,1,-5,2,-4]

[3, -2, 1, -5, 2, -4]


### 14.3. Khi số chẵn và lẻ **không** bằng nhau

Nếu đề bỏ điều kiện "đúng nửa", có thể không tồn tại đáp án. Kiểm tra trước: số số chẵn phải bằng `(n + 1) // 2` (số vị trí chẵn).

In [29]:
def possible(nums):
    return sum(1 for x in nums if x % 2 == 0) == (len(nums) + 1) // 2

print(possible([2, 3, 4]), possible([2, 4, 6, 1]))

True False


## 15. Bài tập tự luyện (TODO — không tự chạy)

### Bài 1 — Đoán output
```python
list(range(1, 9, 2)), [10, 11, 12, 13][1::2], sorted([5, 2, 3, 8], key=lambda x: x % 2)
```

### Bài 2 — Viết validator

In [30]:
def exercise_is_valid(original: list[int], answer: list[int]) -> bool:
    # TODO: hoán vị + mọi i: answer[i] % 2 == i % 2
    pass

# assert exercise_is_valid([4,2,5,7], [2,7,4,5]) == True
# assert exercise_is_valid([4,2,5,7], [4,2,5,7]) == False

### Bài 3 — Hai con trỏ ghi (không tại chỗ)

In [31]:
def exercise_write_pointers(nums: list[int]) -> list[int]:
    # TODO: res = [0]*n, even_idx = 0, odd_idx = 1
    pass

# out = exercise_write_pointers([3,1,4,2,6,5]); assert is_valid([3,1,4,2,6,5], out)

### Bài 4 — Tự viết bản tại chỗ (follow-up)

In [32]:
def exercise_inplace(nums: list[int]) -> list[int]:
    # TODO: i duyệt vị trí chẵn, j (khai báo ngoài vòng) duyệt vị trí lẻ
    pass

# src = [1,3,5,7,0,2,4,6]; assert is_valid(src, exercise_inplace(src[:]))

### Bài 5 — LeetCode 905 Sort Array By Parity

In [33]:
def exercise_parity_905(nums: list[int]) -> list[int]:
    # TODO: số chẵn đứng trước mọi số lẻ
    pass

# out = exercise_parity_905([3,1,2,4]); assert sorted(out) == [1,2,3,4]
# assert all(x % 2 == 0 for x in out[:2]) and all(x % 2 == 1 for x in out[2:])

### Bài 6 — Mini challenge: số lần đổi chỗ ít nhất

Trả về **số lần đổi chỗ ít nhất** để `nums` thỏa điều kiện bài 922 (mỗi lần đổi chỗ hai phần tử bất kỳ).

In [34]:
def exercise_min_swaps(nums: list[int]) -> int:
    # TODO
    pass

# assert exercise_min_swaps([4,2,5,7]) == 1
# assert exercise_min_swaps([1,2,3,4,5,6,7,8]) == 4
# assert exercise_min_swaps([2,3]) == 0

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | range có bước 2 | slice từ index 1, bước 2 | sort ổn định, khóa 0 trước 1 |
| 2 | 2 điều kiện | `sorted(a) == sorted(b)` | `all(x % 2 == i % 2 for i, x in enumerate(answer))` |
| 3 | số chẵn → ô chẵn tiếp theo | 2 biến chỉ số tăng 2 | xem mục 7.1 |
| 4 | đổi chỗ 2 phần tử lạc chỗ | `for i in range(0,n,2)` + `while` | `j` không reset |
| 5 | hai đầu đi vào | `l` tìm lẻ, `r` tìm chẵn | khung `if/elif/else` |
| 6 | mỗi lần đổi sửa được tối đa 2 ô | đếm vị trí chẵn chứa số lẻ | đáp án = số đó |

## 17. Đáp án tham khảo

Bài 1: `([1, 3, 5, 7], [11, 13], [2, 8, 5, 3])`.

In [35]:
print((list(range(1, 9, 2)), [10, 11, 12, 13][1::2], sorted([5, 2, 3, 8], key=lambda x: x % 2)))

([1, 3, 5, 7], [11, 13], [2, 8, 5, 3])


In [36]:
def answer_is_valid(original, answer):
    return sorted(original) == sorted(answer) and all(x % 2 == i % 2 for i, x in enumerate(answer))

def answer_write_pointers(nums):
    res = [0] * len(nums)
    e, o = 0, 1
    for x in nums:
        if x % 2 == 0:
            res[e] = x; e += 2
        else:
            res[o] = x; o += 2
    return res

def answer_inplace(nums):
    j = 1
    for i in range(0, len(nums), 2):
        if nums[i] % 2 == 1:
            while nums[j] % 2 == 1:
                j += 2
            nums[i], nums[j] = nums[j], nums[i]
    return nums

def answer_parity_905(nums):
    l, r = 0, len(nums) - 1
    while l < r:
        if nums[l] % 2 == 0:
            l += 1
        elif nums[r] % 2 == 1:
            r -= 1
        else:
            nums[l], nums[r] = nums[r], nums[l]
            l += 1; r -= 1
    return nums

def answer_min_swaps(nums):
    return sum(1 for i in range(0, len(nums), 2) if nums[i] % 2 == 1)

assert answer_is_valid([4,2,5,7], [2,7,4,5]) == True
assert answer_is_valid([4,2,5,7], [4,2,5,7]) == False
assert is_valid([3,1,4,2,6,5], answer_write_pointers([3,1,4,2,6,5]))
src = [1,3,5,7,0,2,4,6]; assert is_valid(src, answer_inplace(src[:]))
out = answer_parity_905([3,1,2,4]); assert sorted(out) == [1,2,3,4]
assert all(x % 2 == 0 for x in out[:2]) and all(x % 2 == 1 for x in out[2:])
assert answer_min_swaps([4,2,5,7]) == 1
assert answer_min_swaps([1,2,3,4,5,6,7,8]) == 4
assert answer_min_swaps([2,3]) == 0
for nums in tests:
    assert is_valid(nums, answer_inplace(nums[:]))
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


Giải thích Bài 6: mỗi vị trí chẵn chứa số lẻ cần được sửa; số này bằng số vị trí lẻ chứa số chẵn (mục 4). Mỗi lần đổi chỗ sửa **tối đa 2** ô sai (một ô chẵn + một ô lẻ), nên cần ít nhất `k` lần, và thuật toán in-place đạt đúng `k` lần.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def sortArrayByParityII(self, nums: list[int]) -> list[int]:
        j = 1
        for i in range(0, len(nums), 2):
            if nums[i] % 2 == 1:
                while nums[j] % 2 == 1:
                    j += 2
                nums[i], nums[j] = nums[j], nums[i]
        return nums
```

**Câu thần chú:** *"i đi ô chẵn, j đi ô lẻ, gặp lẻ lạc thì tìm chẵn lạc rồi đổi."*

**Checklist trước khi submit:**
- [ ] `i` và `j` đều bước **2**.
- [ ] `j` khai báo **ngoài** vòng `for` (không reset).
- [ ] Test bằng validator, không so với một đáp án cố định.
- [ ] Sao chép input khi test hàm sửa tại chỗ.
- [ ] Biết giải thích vì sao `while` không vượt mảng (đếm số phần tử lạc chỗ).